# Notebook 09 — Copula Dependence and Tail-Risk Robustness

## Objective

This notebook provides an optional robustness extension to the main portfolio
analysis.

The previous notebooks primarily model dependence through covariance and
correlation. These measures describe average co-movement but may not fully
capture dependence during extreme market events.

Copula models allow the marginal return distributions of individual assets
to be separated from their dependence structure.

The analysis focuses on the recommended Risk Parity portfolio:

- AGG
- GLD
- XLK

Two dependence models are compared:

1. Gaussian copula
2. Student-t copula

The Student-t copula can capture joint tail dependence and is therefore useful
for testing whether conventional Gaussian dependence assumptions understate
portfolio downside risk.

This notebook is used as a robustness check. It does not re-optimize the
portfolio or use future information to change the frozen allocation.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    norm,
    t,
    rankdata
)

from scipy.special import gammaln

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    160
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.6f}"
)

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Project root:",
    PROJECT_ROOT.resolve()
)

Project root: C:\quant_edge_round2


In [3]:
simple_returns = pd.read_csv(
    PROCESSED_DIR
    / "simple_returns.csv",
    index_col=0,
    parse_dates=True
)

simple_returns = (
    simple_returns
    .sort_index()
)

print(
    "Full return sample:",
    simple_returns.shape
)

print(
    "Start:",
    simple_returns.index.min()
)

print(
    "End:",
    simple_returns.index.max()
)

Full return sample: (5512, 7)
Start: 2004-11-19 00:00:00
End: 2026-10-19 00:00:00


In [4]:
TRAIN_END = "2021-12-31"

training_returns = (
    simple_returns
    .loc[
        simple_returns.index
        <= TRAIN_END
    ]
    .copy()
)

print(
    "Training observations:",
    len(training_returns)
)

print(
    "Training end:",
    training_returns.index.max()
)

Training observations: 4309
Training end: 2021-12-31 00:00:00


In [5]:
recommended_allocation = pd.read_csv(
    TABLE_DIR
    / "recommended_portfolio_allocation.csv"
)

copula_assets = (
    recommended_allocation[
        "Asset"
    ]
    .tolist()
)

copula_weights = (
    recommended_allocation[
        "Weight"
    ]
    .to_numpy(
        dtype=float
    )
)

print(
    "Assets:",
    copula_assets
)

print(
    "Weights:",
    copula_weights
)

print(
    "Weight sum:",
    copula_weights.sum()
)

assert set(
    copula_assets
) == {
    "AGG",
    "GLD",
    "XLK"
}

assert np.isclose(
    copula_weights.sum(),
    1.0,
    atol=1e-6
)

print(
    "Notebook 08 -> Notebook 09 "
    "recommendation handoff validated."
)

Assets: ['AGG', 'GLD', 'XLK']
Weights: [0.65454107 0.17790722 0.16755171]
Weight sum: 1.0
Notebook 08 -> Notebook 09 recommendation handoff validated.


In [6]:
copula_returns = (
    training_returns[
        copula_assets
    ]
    .dropna()
    .copy()
)

print(
    "Copula return observations:",
    len(copula_returns)
)

print(
    "Copula sample period:",
    copula_returns.index.min(),
    "to",
    copula_returns.index.max()
)

copula_returns.head()

Copula return observations: 4309
Copula sample period: 2004-11-19 00:00:00 to 2021-12-31 00:00:00


,AGG,GLD,XLK
2004-11-19,-0.002044,0.009013,-0.015464
2004-11-22,0.001561,0.003796,0.001428
2004-11-23,-0.000585,-0.004449,-0.004278
2004-11-24,0.000974,0.006704,0.008115
2004-11-26,-0.002920,0.005327,-0.001420


## 1. Dependence Diagnostics

Pearson correlation measures linear dependence.

Spearman correlation and Kendall's tau are rank-based measures and are useful
for copula modelling because copulas describe dependence independently of the
marginal return scales.

If dependence becomes stronger during extreme losses, ordinary linear
correlation may understate portfolio tail risk.

In [7]:
pearson_corr = (
    copula_returns
    .corr(
        method="pearson"
    )
)

pearson_corr.round(4)

,AGG,GLD,XLK
AGG,1.000000,0.172500,-0.078300
GLD,0.172500,1.000000,0.028200
XLK,-0.078300,0.028200,1.000000


In [8]:
spearman_corr = (
    copula_returns
    .corr(
        method="spearman"
    )
)

spearman_corr.round(4)

,AGG,GLD,XLK
AGG,1.000000,0.200800,-0.169600
GLD,0.200800,1.000000,0.026500
XLK,-0.169600,0.026500,1.000000


In [9]:
kendall_corr = (
    copula_returns
    .corr(
        method="kendall"
    )
)

kendall_corr.round(4)

,AGG,GLD,XLK
AGG,1.000000,0.140200,-0.117000
GLD,0.140200,1.000000,0.018000
XLK,-0.117000,0.018000,1.000000


## 2. Pseudo-Observation Transformation

Copulas model the dependence between uniform variables.

For each asset return observation \(r_{it}\), the empirical distribution is
converted into a pseudo-observation:

\[
u_{it}
=
\frac{\operatorname{rank}(r_{it})}{n+1}
\]

This produces values strictly between zero and one and avoids imposing a
parametric distribution on the individual asset marginals.

The approach is therefore semi-parametric:

- dependence is modelled parametrically through a copula;
- individual asset return distributions remain empirical.

In [10]:
n_obs = len(
    copula_returns
)

pseudo_observations = pd.DataFrame(
    index=copula_returns.index,
    columns=copula_assets,
    dtype=float
)

for asset in copula_assets:

    pseudo_observations[
        asset
    ] = (
        rankdata(
            copula_returns[
                asset
            ],
            method="average"
        )
        / (
            n_obs
            + 1
        )
    )

pseudo_observations.head()

,AGG,GLD,XLK
2004-11-19,0.145708,0.828306,0.082367
2004-11-22,0.750348,0.659397,0.516009
2004-11-23,0.352436,0.275406,0.270070
2004-11-24,0.649188,0.766821,0.786543
2004-11-26,0.083759,0.721578,0.380742


In [11]:
print(
    "Minimum pseudo-observation:"
)

print(
    pseudo_observations.min()
)

print(
    "\nMaximum pseudo-observation:"
)

print(
    pseudo_observations.max()
)

assert (
    (
        pseudo_observations
        > 0
    )
    .all()
    .all()
)

assert (
    (
        pseudo_observations
        < 1
    )
    .all()
    .all()
)

print(
    "\nPseudo-observations validated."
)

Minimum pseudo-observation:
AGG   0.000232
GLD   0.000232
XLK   0.000232
dtype: float64

Maximum pseudo-observation:
AGG   0.999768
GLD   0.999768
XLK   0.999768
dtype: float64

Pseudo-observations validated.


## 3. Gaussian Copula

The Gaussian copula models dependence using a latent multivariate normal
distribution.

The pseudo-observations are transformed using

\[
z_i = \Phi^{-1}(u_i)
\]

and the correlation matrix of the latent variables is estimated.

The Gaussian copula can reproduce ordinary dependence patterns but has zero
asymptotic tail dependence when correlations are less than one.

It therefore serves as the benchmark copula model.

In [12]:
gaussian_latent = pd.DataFrame(
    norm.ppf(
        pseudo_observations
    ),
    index=pseudo_observations.index,
    columns=copula_assets
)

gaussian_corr = (
    gaussian_latent.corr()
)

gaussian_corr.round(4)

,AGG,GLD,XLK
AGG,1.000000,0.191200,-0.164100
GLD,0.191200,1.000000,0.027000
XLK,-0.164100,0.027000,1.000000


In [13]:
gaussian_eigenvalues = (
    np.linalg.eigvalsh(
        gaussian_corr.values
    )
)

print(
    "Gaussian copula eigenvalues:",
    gaussian_eigenvalues
)

assert (
    gaussian_eigenvalues
    > 0
).all()

print(
    "Gaussian copula correlation "
    "matrix is positive definite."
)

Gaussian copula eigenvalues: [0.73428047 1.02670603 1.2390135 ]
Gaussian copula correlation matrix is positive definite.


## 4. Student-t Copula

The Student-t copula is used as a heavier-tailed alternative to the Gaussian
copula.

Unlike the Gaussian copula, the Student-t copula allows non-zero asymptotic
tail dependence, meaning extreme asset movements can occur jointly with a
higher probability.

The Student-t copula degrees-of-freedom parameter is selected using a
grid-based pseudo-likelihood comparison.

For each candidate degrees of freedom, the latent Student-t variables are
constructed from the pseudo-observations and their correlation matrix is
estimated from the transformed data.

The candidate producing the highest copula pseudo-log-likelihood is retained.

This is a practical profile-style approximation rather than a full joint
maximum-likelihood optimization of all copula parameters.

In [14]:
def t_copula_loglikelihood(
    u,
    corr,
    df
):

    x = t.ppf(
        u,
        df=df
    )

    d = x.shape[1]

    corr_inv = np.linalg.inv(
        corr
    )

    corr_det = np.linalg.det(
        corr
    )

    quadratic = np.sum(
        x
        @ corr_inv
        * x,
        axis=1
    )

    log_multivariate_t = (
        gammaln(
            (df + d) / 2
        )
        - gammaln(
            df / 2
        )
        - 0.5
        * np.log(
            corr_det
        )
        - (
            d / 2
        )
        * np.log(
            df * np.pi
        )
        - (
            (df + d) / 2
        )
        * np.log1p(
            quadratic
            / df
        )
    )

    log_univariate_t = np.sum(
        t.logpdf(
            x,
            df=df
        ),
        axis=1
    )

    copula_log_density = (
        log_multivariate_t
        - log_univariate_t
    )

    return (
        copula_log_density.sum()
    )

In [15]:
df_candidates = np.arange(
    2.5,
    31.0,
    0.5
)

t_copula_results = []

u_values = (
    pseudo_observations
    .values
)

for df in df_candidates:

    latent_t = (
        t.ppf(
            u_values,
            df=df
        )
    )

    corr_t = np.corrcoef(
        latent_t,
        rowvar=False
    )

    loglik = (
        t_copula_loglikelihood(
            u_values,
            corr_t,
            df
        )
    )

    t_copula_results.append(
        {
            "Degrees of Freedom":
                df,

            "Log Likelihood":
                loglik
        }
    )

t_copula_fit = pd.DataFrame(
    t_copula_results
)

t_copula_fit.head()

,Degrees of Freedom,Log Likelihood
0,2.500000,325.837539
1,3.000000,388.093696
2,3.500000,413.878337
3,4.000000,422.759633
4,4.500000,423.295685


In [16]:
best_t_row = (
    t_copula_fit
    .loc[
        t_copula_fit[
            "Log Likelihood"
        ].idxmax()
    ]
)

best_df = float(
    best_t_row[
        "Degrees of Freedom"
    ]
)

print(
    "Selected Student-t degrees of freedom:",
    best_df
)

print(
    "Maximum pseudo-log-likelihood:",
    best_t_row[
        "Log Likelihood"
    ]
)

Selected Student-t degrees of freedom: 4.5
Maximum pseudo-log-likelihood: 423.2956852504011


In [17]:
t_latent = (
    t.ppf(
        pseudo_observations.values,
        df=best_df
    )
)

t_copula_corr = pd.DataFrame(
    np.corrcoef(
        t_latent,
        rowvar=False
    ),
    index=copula_assets,
    columns=copula_assets
)

t_copula_corr.round(4)

,AGG,GLD,XLK
AGG,1.000000,0.178100,-0.140800
GLD,0.178100,1.000000,0.029300
XLK,-0.140800,0.029300,1.000000


In [18]:
plt.figure(
    figsize=(9, 5)
)

plt.plot(
    t_copula_fit[
        "Degrees of Freedom"
    ],
    t_copula_fit[
        "Log Likelihood"
    ]
)

plt.axvline(
    best_df,
    linestyle="--"
)

plt.xlabel(
    "Student-t Degrees of Freedom"
)

plt.ylabel(
    "Copula Pseudo-Log-Likelihood"
)

plt.title(
    "Student-t Copula Degrees-of-Freedom Selection"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "t_copula_df_selection.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_18692\4090877475.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Implied Tail Dependence

For a Gaussian copula, asymptotic lower-tail dependence is zero unless
correlation is perfect.

For a Student-t copula, pairwise lower-tail dependence is

\[
\lambda_L
=
2t_{\nu+1}
\left(
-
\sqrt{
\frac{(\nu+1)(1-\rho)}
{1+\rho}
}
\right)
\]

where \(ρ\) is the pairwise copula correlation and \(ν\) is the estimated
degrees of freedom.

A larger value indicates a greater probability that two assets experience
extreme losses together.

In [19]:
def t_tail_dependence(
    rho,
    df
):

    value = (
        2
        * t.cdf(
            -np.sqrt(
                (
                    (df + 1)
                    * (1 - rho)
                )
                /
                (1 + rho)
            ),
            df=df + 1
        )
    )

    return value

In [20]:
tail_dependence_rows = []

for i in range(
    len(copula_assets)
):

    for j in range(
        i + 1,
        len(copula_assets)
    ):

        asset_1 = (
            copula_assets[i]
        )

        asset_2 = (
            copula_assets[j]
        )

        rho = (
            t_copula_corr.loc[
                asset_1,
                asset_2
            ]
        )

        lambda_tail = (
            t_tail_dependence(
                rho,
                best_df
            )
        )

        tail_dependence_rows.append(
            {
                "Asset 1":
                    asset_1,

                "Asset 2":
                    asset_2,

                "t-Copula Correlation":
                    rho,

                "Gaussian Tail Dependence":
                    0.0,

                "Student-t Tail Dependence":
                    lambda_tail
            }
        )

tail_dependence_table = pd.DataFrame(
    tail_dependence_rows
)

tail_dependence_table.round(4)

,Asset 1,Asset 2,t-Copula Correlation,Gaussian Tail Dependence,Student-t Tail Dependence
0,AGG,GLD,0.178100,0.000000,0.102200
1,AGG,XLK,-0.140800,0.000000,0.038700
2,GLD,XLK,0.029300,0.000000,0.067000


## 6. Copula-Based Portfolio Simulation

The Gaussian and Student-t copulas are now used to generate joint dependence
scenarios for AGG, GLD and XLK.

To avoid imposing a parametric distribution on each asset's marginal returns,
the simulated copula probabilities are mapped back to returns using the
empirical training-period marginal distributions.

This creates a semi-parametric simulation framework:

- empirical marginal return distributions,
- Gaussian or Student-t dependence structure,
- frozen Risk Parity portfolio weights.

The resulting portfolio distributions are used to estimate 95% and 99% VaR
and Expected Shortfall.

In [21]:
COPULA_SIMULATIONS = 100_000
COPULA_RANDOM_SEED = 42

rng = np.random.default_rng(
    COPULA_RANDOM_SEED
)

print(
    "Copula simulations:",
    COPULA_SIMULATIONS
)

print(
    "Random seed:",
    COPULA_RANDOM_SEED
)

Copula simulations: 100000
Random seed: 42


In [22]:
def empirical_inverse_cdf(
    uniform_values,
    historical_returns
):

    uniform_values = np.clip(
        uniform_values,
        1e-6,
        1 - 1e-6
    )

    return np.quantile(
        historical_returns,
        uniform_values
    )

In [23]:
t_copula_eigenvalues = (
    np.linalg.eigvalsh(
        t_copula_corr.values
    )
)

print(
    "Student-t copula eigenvalues:",
    t_copula_eigenvalues
)

assert (
    t_copula_eigenvalues
    > 0
).all()

print(
    "Student-t copula correlation "
    "matrix is positive definite."
)

Student-t copula eigenvalues: [0.75819372 1.02851861 1.21328766]
Student-t copula correlation matrix is positive definite.


In [24]:
def uniforms_to_empirical_returns(
    uniform_matrix,
    historical_data,
    assets
):

    simulated_returns = np.zeros_like(
        uniform_matrix,
        dtype=float
    )

    for j, asset in enumerate(
        assets
    ):

        simulated_returns[
            :,
            j
        ] = empirical_inverse_cdf(
            uniform_matrix[
                :,
                j
            ],
            historical_data[
                asset
            ].values
        )

    return simulated_returns

In [25]:
gaussian_latent_sim = (
    rng.multivariate_normal(
        mean=np.zeros(
            len(copula_assets)
        ),
        cov=gaussian_corr.values,
        size=COPULA_SIMULATIONS
    )
)

gaussian_uniform_sim = (
    norm.cdf(
        gaussian_latent_sim
    )
)

print(
    gaussian_uniform_sim.shape
)

(100000, 3)


In [26]:
gaussian_asset_returns = (
    uniforms_to_empirical_returns(
        gaussian_uniform_sim,
        copula_returns,
        copula_assets
    )
)

gaussian_asset_returns = pd.DataFrame(
    gaussian_asset_returns,
    columns=copula_assets
)

gaussian_asset_returns.describe()

,AGG,GLD,XLK
count,"100,000.000000","100,000.000000","100,000.000000"
mean,0.000171,0.000370,0.000639
std,0.002956,0.011247,0.013378
min,-0.067639,-0.087750,-0.136322
25%,-0.001191,-0.005103,-0.004971
50%,0.000192,0.000564,0.001041
75%,0.001586,0.006169,0.006848
max,0.038628,0.112443,0.138820


In [27]:
gaussian_portfolio_returns = (
    gaussian_asset_returns.values
    @ copula_weights
)

pd.Series(
    gaussian_portfolio_returns
).describe()

count   100,000.000000
mean          0.000285
std           0.003607
min          -0.052636
25%          -0.001659
50%           0.000360
75%           0.002323
max           0.031554
dtype: float64

In [28]:
rng_t = np.random.default_rng(
    COPULA_RANDOM_SEED
)

normal_draws = (
    rng_t.multivariate_normal(
        mean=np.zeros(
            len(copula_assets)
        ),
        cov=t_copula_corr.values,
        size=COPULA_SIMULATIONS
    )
)

chi_square_draws = (
    rng_t.chisquare(
        df=best_df,
        size=COPULA_SIMULATIONS
    )
)

t_latent_sim = (
    normal_draws
    /
    np.sqrt(
        chi_square_draws[
            :,
            None
        ]
        / best_df
    )
)

t_uniform_sim = (
    t.cdf(
        t_latent_sim,
        df=best_df
    )
)

print(
    t_uniform_sim.shape
)

(100000, 3)


In [29]:
t_asset_returns = (
    uniforms_to_empirical_returns(
        t_uniform_sim,
        copula_returns,
        copula_assets
    )
)

t_asset_returns = pd.DataFrame(
    t_asset_returns,
    columns=copula_assets
)

t_asset_returns.describe()

,AGG,GLD,XLK
count,"100,000.000000","100,000.000000","100,000.000000"
mean,0.000158,0.000414,0.000551
std,0.003001,0.011308,0.013415
min,-0.065827,-0.087161,-0.137965
25%,-0.001195,-0.005076,-0.005022
50%,0.000188,0.000557,0.000972
75%,0.001570,0.006169,0.006745
max,0.038628,0.112736,0.138191


In [30]:
t_portfolio_returns = (
    t_asset_returns.values
    @ copula_weights
)

pd.Series(
    t_portfolio_returns
).describe()

count   100,000.000000
mean          0.000270
std           0.003652
min          -0.075189
25%          -0.001482
50%           0.000334
75%           0.002122
max           0.042447
dtype: float64

## 7. Copula-Based VaR and Expected Shortfall

The simulated portfolio return distributions are used to estimate daily
Value-at-Risk and Expected Shortfall at the 95% and 99% confidence levels.

For confidence level \(q\):

\[
VaR_q = -Q_{1-q}(R_p)
\]

where \(Q_{1-q}\) is the lower-tail portfolio return quantile.

Expected Shortfall is the average loss conditional on returns falling at or
below the corresponding VaR threshold:

\[
ES_q
=
-E[
R_p
\mid
R_p \le Q_{1-q}(R_p)
]
\]

The Gaussian and Student-t copula results are compared using the same
portfolio weights, empirical marginal distributions and number of simulations,
so differences primarily reflect the assumed dependence structure.`

In [31]:
def calculate_var_es(
    portfolio_returns,
    confidence
):

    portfolio_returns = np.asarray(
        portfolio_returns
    )

    cutoff = np.quantile(
        portfolio_returns,
        1 - confidence
    )

    var = -cutoff

    tail_returns = (
        portfolio_returns[
            portfolio_returns
            <= cutoff
        ]
    )

    es = (
        -tail_returns.mean()
    )

    return var, es

In [32]:
confidence_levels = [
    0.95,
    0.99
]

copula_risk_rows = []

for confidence in confidence_levels:

    gaussian_var, gaussian_es = (
        calculate_var_es(
            gaussian_portfolio_returns,
            confidence
        )
    )

    t_var, t_es = (
        calculate_var_es(
            t_portfolio_returns,
            confidence
        )
    )

    copula_risk_rows.extend(
        [
            {
                "Model":
                    "Gaussian Copula",

                "Confidence":
                    confidence,

                "VaR":
                    gaussian_var,

                "ES":
                    gaussian_es
            },

            {
                "Model":
                    "Student-t Copula",

                "Confidence":
                    confidence,

                "VaR":
                    t_var,

                "ES":
                    t_es
            }
        ]
    )

copula_risk_results = pd.DataFrame(
    copula_risk_rows
)

copula_risk_results

,Model,Confidence,VaR,ES
0,Gaussian Copula,0.950000,0.005412,0.008021
1,Student-t Copula,0.950000,0.005232,0.008315
2,Gaussian Copula,0.990000,0.009353,0.012847
3,Student-t Copula,0.990000,0.009632,0.014426


In [33]:
copula_risk_display = (
    copula_risk_results.copy()
)

copula_risk_display[
    "Confidence"
] *= 100

copula_risk_display[
    "VaR"
] *= 100

copula_risk_display[
    "ES"
] *= 100

copula_risk_display.round(3)

,Model,Confidence,VaR,ES
0,Gaussian Copula,95.000000,0.541000,0.802000
1,Student-t Copula,95.000000,0.523000,0.832000
2,Gaussian Copula,99.000000,0.935000,1.285000
3,Student-t Copula,99.000000,0.963000,1.443000


In [34]:
copula_var_comparison = (
    copula_risk_results
    .pivot(
        index="Confidence",
        columns="Model",
        values="VaR"
    )
)

copula_es_comparison = (
    copula_risk_results
    .pivot(
        index="Confidence",
        columns="Model",
        values="ES"
    )
)

print(
    "VaR comparison (%)"
)

display(
    (
        copula_var_comparison
        * 100
    ).round(3)
)

print(
    "\nES comparison (%)"
)

display(
    (
        copula_es_comparison
        * 100
    ).round(3)
)

VaR comparison (%)


Model,Gaussian Copula,Student-t Copula
Confidence,,
0.950000,0.541000,0.523000
0.990000,0.935000,0.963000



ES comparison (%)


Model,Gaussian Copula,Student-t Copula
Confidence,,
0.950000,0.802000,0.832000
0.990000,1.285000,1.443000


In [35]:
tail_risk_uplift = pd.DataFrame(
    index=confidence_levels
)

tail_risk_uplift[
    "VaR Uplift"
] = (
    (
        copula_var_comparison[
            "Student-t Copula"
        ]
        /
        copula_var_comparison[
            "Gaussian Copula"
        ]
    )
    - 1
)

tail_risk_uplift[
    "ES Uplift"
] = (
    (
        copula_es_comparison[
            "Student-t Copula"
        ]
        /
        copula_es_comparison[
            "Gaussian Copula"
        ]
    )
    - 1
)

(
    tail_risk_uplift
    * 100
).round(2)

,VaR Uplift,ES Uplift
0.950000,-3.330000,3.670000
0.990000,2.980000,12.300000


In [36]:
plt.figure(
    figsize=(10, 6)
)

plt.hist(
    -gaussian_portfolio_returns,
    bins=150,
    density=True,
    alpha=0.5,
    label="Gaussian Copula"
)

plt.hist(
    -t_portfolio_returns,
    bins=150,
    density=True,
    alpha=0.5,
    label="Student-t Copula"
)

plt.xlim(
    0,
    np.quantile(
        -t_portfolio_returns,
        0.999
    )
)

plt.xlabel(
    "Daily Portfolio Loss"
)

plt.ylabel(
    "Density"
)

plt.title(
    "Gaussian versus Student-t Copula Portfolio Loss Distribution"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "copula_loss_distribution_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_18692\3298112186.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
plot_table = (
    copula_risk_results
    .pivot(
        index="Model",
        columns="Confidence",
        values=[
            "VaR",
            "ES"
        ]
    )
    * 100
)

plot_table

VaR                ES         
Confidence       0.950000 0.990000 0.950000 0.990000
Model                                               
Gaussian Copula  0.541228 0.935304 0.802110 1.284655
Student-t Copula 0.523226 0.963195 0.831524 1.442610

In [38]:
plot_table.columns = [
    f"{metric} {confidence:.0f}%"
    for metric, confidence
    in plot_table.columns
]

In [39]:
plot_table.plot(
    kind="bar",
    figsize=(11, 7)
)

plt.ylabel(
    "Daily Loss (%)"
)

plt.title(
    "Copula-Based VaR and Expected Shortfall"
)

plt.xticks(
    rotation=0
)

plt.legend(
    title=None
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "copula_var_es_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_18692\1937465825.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Copula Model-Fit Comparison

Gaussian and Student-t dependence models are compared using their
pseudo-log-likelihood values.

A pseudo-AIC measure is also reported:

\[
\text{Pseudo-AIC}
=
2k - 2\ell
\]

where \(k\) is the number of estimated copula parameters and \(\ell\) is the
pseudo-log-likelihood.

Because the models are estimated using pseudo-likelihood rather than a full
joint likelihood, this measure is interpreted as a comparative diagnostic
rather than an exact classical AIC.

In [40]:
def gaussian_copula_loglikelihood(
    u,
    corr
):

    z = norm.ppf(
        u
    )

    inv_corr = np.linalg.inv(
        corr
    )

    identity = np.eye(
        corr.shape[0]
    )

    log_det = np.log(
        np.linalg.det(
            corr
        )
    )

    quadratic = np.sum(
        z
        @ (
            inv_corr
            - identity
        )
        * z,
        axis=1
    )

    log_density = (
        -0.5
        * log_det
        -0.5
        * quadratic
    )

    return log_density.sum()

In [41]:
gaussian_loglik = (
    gaussian_copula_loglikelihood(
        pseudo_observations.values,
        gaussian_corr.values
    )
)

t_loglik = (
    best_t_row[
        "Log Likelihood"
    ]
)

copula_model_comparison = pd.DataFrame(
    {
        "Model": [
            "Gaussian Copula",
            "Student-t Copula"
        ],

        "Log Likelihood": [
            gaussian_loglik,
            t_loglik
        ]
    }
)

copula_model_comparison

,Model,Log Likelihood
0,Gaussian Copula,146.921957
1,Student-t Copula,423.295685


In [42]:
n_corr_parameters = (
    len(copula_assets)
    * (
        len(copula_assets)
        - 1
    )
    // 2
)

gaussian_parameters = (
    n_corr_parameters
)

t_parameters = (
    n_corr_parameters
    + 1
)

copula_model_comparison[
    "Parameters"
] = [
    gaussian_parameters,
    t_parameters
]

copula_model_comparison[
    "Pseudo-AIC"
] = (
    2
    * copula_model_comparison[
        "Parameters"
    ]
    -
    2
    * copula_model_comparison[
        "Log Likelihood"
    ]
)

copula_model_comparison

,Model,Log Likelihood,Parameters,Pseudo-AIC
0,Gaussian Copula,146.921957,3,-287.843914
1,Student-t Copula,423.295685,4,-838.591371


## Copula Interpretation

The fitted Student-t copula has approximately 4.5 degrees of freedom,
indicating materially heavier joint tails than the Gaussian dependence model.

The estimated lower-tail dependence coefficients are positive for all three
asset pairs.

The strongest estimated pairwise lower-tail dependence occurs between AGG
and GLD, while AGG and XLK display the weakest tail dependence.

These results show that low ordinary correlation does not imply that assets
are completely independent during extreme market outcomes.

The Gaussian copula assumes zero asymptotic tail dependence, whereas the
Student-t copula allows extreme outcomes to occur jointly.

Consequently, differences between Gaussian-copula and Student-t-copula VaR
or Expected Shortfall provide a direct measure of dependence-model risk.

## Limitations of the Copula Extension

This copula analysis is an optional robustness extension rather than the
primary portfolio construction framework.

Several limitations should be recognized.

1. Dependence parameters are estimated from historical data and may change
   through time.

2. The Student-t copula assumes symmetric upper- and lower-tail dependence.

3. Empirical marginal distributions preserve historical skewness and
   kurtosis but cannot generate asset-specific returns outside the historical
   observed range.

4. Copula simulation does not replace the separate EVT analysis, which is
   specifically designed to model losses beyond high thresholds.

5. The analysis is performed on the frozen recommended portfolio and is not
   used to re-optimize weights using future information.

The copula model is therefore interpreted as an additional robustness check
on dependence and tail-risk assumptions.

In [43]:
assert np.isclose(
    copula_weights.sum(),
    1.0
)

assert (
    gaussian_uniform_sim
    > 0
).all()

assert (
    gaussian_uniform_sim
    < 1
).all()

assert (
    t_uniform_sim
    > 0
).all()

assert (
    t_uniform_sim
    < 1
).all()

assert (
    copula_risk_results[
        "ES"
    ]
    >= copula_risk_results[
        "VaR"
    ]
).all()

assert (
    best_df
    > 2
)

print(
    "All copula simulation validation checks passed."
)

All copula simulation validation checks passed.


In [44]:
pearson_corr.to_csv(
    TABLE_DIR
    / "copula_pearson_correlation.csv"
)

spearman_corr.to_csv(
    TABLE_DIR
    / "copula_spearman_correlation.csv"
)

kendall_corr.to_csv(
    TABLE_DIR
    / "copula_kendall_correlation.csv"
)

gaussian_corr.to_csv(
    TABLE_DIR
    / "gaussian_copula_correlation.csv"
)

t_copula_corr.to_csv(
    TABLE_DIR
    / "student_t_copula_correlation.csv"
)

t_copula_fit.to_csv(
    TABLE_DIR
    / "student_t_df_selection.csv",
    index=False
)

tail_dependence_table.to_csv(
    TABLE_DIR
    / "copula_tail_dependence.csv",
    index=False
)

copula_risk_results.to_csv(
    TABLE_DIR
    / "copula_var_es_results.csv",
    index=False
)

tail_risk_uplift.to_csv(
    TABLE_DIR
    / "copula_tail_risk_uplift.csv"
)

copula_model_comparison.to_csv(
    TABLE_DIR
    / "copula_model_comparison.csv",
    index=False
)

print(
    "Notebook 09 outputs saved."
)

Notebook 09 outputs saved.


## Tail-Risk Robustness Result

The Gaussian and Student-t copulas produce relatively similar risk estimates
at the 95% confidence level.

At the 99% level, however, the effect of tail dependence becomes more
important.

The Student-t copula produces approximately:

- 2.99% higher 99% VaR, and
- 12.30% higher 99% Expected Shortfall

than the Gaussian copula.

The substantially larger increase in Expected Shortfall indicates that
dependence-model choice matters most for the severity of losses beyond the
VaR threshold rather than for moderate quantiles of the return distribution.

This is particularly relevant to the endowment mandate because capital
preservation depends not only on the probability of crossing a loss threshold
but also on the magnitude of losses once that threshold has been exceeded.

## Copula Model Selection

The Student-t copula achieves a substantially higher pseudo-log-likelihood
than the Gaussian copula.

Its pseudo-AIC is also materially lower under this estimation framework, even after accounting for the additional
degrees-of-freedom parameter.

This provides empirical support for using the Student-t copula as the more
appropriate dependence robustness model for the recommended portfolio.

The result does not imply that the Gaussian copula is invalid. Instead, it
shows that a dependence structure allowing joint heavy-tail behavior provides
a better fit to the training-period data.

In [45]:
normal_mc_summary = pd.read_csv(
    TABLE_DIR
    / "monte_carlo_var_es_summary.csv"
)

normal_mc_risk_parity = (
    normal_mc_summary[
        normal_mc_summary[
            "Portfolio"
        ]
        == "Best Risk Parity"
    ]
    .iloc[0]
)

normal_mc_risk_parity

Portfolio              Best Risk Parity
Monte Carlo VaR 90%            0.004538
Monte Carlo VaR 95%            0.005905
Monte Carlo VaR 99%            0.008485
Monte Carlo ES 90%             0.006334
Monte Carlo ES 95%             0.007505
Monte Carlo ES 99%             0.009756
Name: 3, dtype: object

In [46]:
three_model_comparison = pd.DataFrame(
    {
        "Model": [
            "Normal Monte Carlo",
            "Gaussian Copula",
            "Student-t Copula"
        ],

        "VaR 95%": [
            normal_mc_risk_parity[
                "Monte Carlo VaR 95%"
            ],
            copula_var_comparison.loc[
                0.95,
                "Gaussian Copula"
            ],
            copula_var_comparison.loc[
                0.95,
                "Student-t Copula"
            ]
        ],

        "VaR 99%": [
            normal_mc_risk_parity[
                "Monte Carlo VaR 99%"
            ],
            copula_var_comparison.loc[
                0.99,
                "Gaussian Copula"
            ],
            copula_var_comparison.loc[
                0.99,
                "Student-t Copula"
            ]
        ],

        "ES 95%": [
            normal_mc_risk_parity[
                "Monte Carlo ES 95%"
            ],
            copula_es_comparison.loc[
                0.95,
                "Gaussian Copula"
            ],
            copula_es_comparison.loc[
                0.95,
                "Student-t Copula"
            ]
        ],

        "ES 99%": [
            normal_mc_risk_parity[
                "Monte Carlo ES 99%"
            ],
            copula_es_comparison.loc[
                0.99,
                "Gaussian Copula"
            ],
            copula_es_comparison.loc[
                0.99,
                "Student-t Copula"
            ]
        ]
    }
)

three_model_comparison

,Model,VaR 95%,VaR 99%,ES 95%,ES 99%
0,Normal Monte Carlo,0.005905,0.008485,0.007505,0.009756
1,Gaussian Copula,0.005412,0.009353,0.008021,0.012847
2,Student-t Copula,0.005232,0.009632,0.008315,0.014426


In [47]:
three_model_display = (
    three_model_comparison.copy()
)

for col in [
    "VaR 95%",
    "VaR 99%",
    "ES 95%",
    "ES 99%"
]:
    three_model_display[
        col
    ] *= 100

three_model_display.round(3)

,Model,VaR 95%,VaR 99%,ES 95%,ES 99%
0,Normal Monte Carlo,0.590000,0.849000,0.751000,0.976000
1,Gaussian Copula,0.541000,0.935000,0.802000,1.285000
2,Student-t Copula,0.523000,0.963000,0.832000,1.443000


In [48]:
student_t_vs_normal_mc = pd.Series(
    {
        "VaR 95% Uplift":
            (
                three_model_comparison.loc[
                    three_model_comparison[
                        "Model"
                    ]
                    == "Student-t Copula",
                    "VaR 95%"
                ].iloc[0]
                /
                three_model_comparison.loc[
                    three_model_comparison[
                        "Model"
                    ]
                    == "Normal Monte Carlo",
                    "VaR 95%"
                ].iloc[0]
                - 1
            ),

        "VaR 99% Uplift":
            (
                three_model_comparison.loc[
                    three_model_comparison[
                        "Model"
                    ]
                    == "Student-t Copula",
                    "VaR 99%"
                ].iloc[0]
                /
                three_model_comparison.loc[
                    three_model_comparison[
                        "Model"
                    ]
                    == "Normal Monte Carlo",
                    "VaR 99%"
                ].iloc[0]
                - 1
            ),

        "ES 95% Uplift":
            (
                three_model_comparison.loc[
                    three_model_comparison[
                        "Model"
                    ]
                    == "Student-t Copula",
                    "ES 95%"
                ].iloc[0]
                /
                three_model_comparison.loc[
                    three_model_comparison[
                        "Model"
                    ]
                    == "Normal Monte Carlo",
                    "ES 95%"
                ].iloc[0]
                - 1
            ),

        "ES 99% Uplift":
            (
                three_model_comparison.loc[
                    three_model_comparison[
                        "Model"
                    ]
                    == "Student-t Copula",
                    "ES 99%"
                ].iloc[0]
                /
                three_model_comparison.loc[
                    three_model_comparison[
                        "Model"
                    ]
                    == "Normal Monte Carlo",
                    "ES 99%"
                ].iloc[0]
                - 1
            )
    }
)

(
    student_t_vs_normal_mc
    * 100
).round(2)

VaR 95% Uplift   -11.390000
VaR 99% Uplift    13.510000
ES 95% Uplift     10.790000
ES 99% Uplift     47.860000
dtype: float64

## Comparison with Conventional Normal Monte Carlo

The copula results were also compared with the multivariate-normal Monte Carlo
model used in Notebook 05.

The conventional Monte Carlo model assumes both normally distributed asset
returns and dependence summarized by the covariance matrix.

The Gaussian copula relaxes the normal marginal-distribution assumption by
retaining empirical asset marginals, while still using Gaussian dependence.

The Student-t copula additionally allows joint tail dependence.

The largest difference appears in 99% Expected Shortfall.

The Student-t copula produces materially higher extreme-loss severity than
the conventional normal Monte Carlo model.

This suggests that a model based only on normal marginals and covariance may
understate the severity of losses in the extreme tail.

The result does not invalidate the original Monte Carlo model. Instead, it
quantifies dependence and distributional model risk and demonstrates why
multiple tail-risk methods are useful for a conservative institutional
investor.

In [49]:
three_model_comparison.to_csv(
    TABLE_DIR
    / "normal_mc_vs_copula_risk_comparison.csv",
    index=False
)

student_t_vs_normal_mc.to_csv(
    TABLE_DIR
    / "student_t_vs_normal_mc_uplift.csv"
)

In [50]:
assert (
    three_model_comparison[
        [
            "VaR 95%",
            "VaR 99%",
            "ES 95%",
            "ES 99%"
        ]
    ]
    > 0
).all().all()

assert (
    three_model_comparison[
        "ES 99%"
    ]
    >=
    three_model_comparison[
        "VaR 99%"
    ]
).all()

print(
    "Three-model tail-risk comparison validated."
)

Three-model tail-risk comparison validated.


# Final Copula Robustness Conclusion

The copula analysis provides additional evidence that dependence assumptions
matter most in the extreme tail of the recommended Risk Parity portfolio.

The Student-t copula is better supported by the training-period dependence
data than the Gaussian copula based on both pseudo-log-likelihood and pseudo-AIC.
At the 95% confidence level, differences between models are relatively small.
However, the gap becomes materially larger at the 99% tail.

Compared with conventional multivariate-normal Monte Carlo, the Student-t
copula produces approximately:

- 13.5% higher 99% VaR,
- 47.9% higher 99% Expected Shortfall.

Compared with the Gaussian copula, the Student-t copula produces approximately:

- 3.0% higher 99% VaR,
- 12.3% higher 99% Expected Shortfall.

This shows that both non-normal marginal returns and joint tail dependence
contribute to model risk.

The result does not change the strategic Risk Parity recommendation.
Instead, it strengthens the case for monitoring the portfolio using multiple
tail-risk models rather than relying exclusively on covariance-based or
normal-distribution assumptions.

The copula extension is therefore treated as a robustness and model-risk
assessment rather than as an additional portfolio-optimization rule.

In [51]:
assert training_returns.shape == (
    4309,
    7
)

assert (
    training_returns.index.max()
    == pd.Timestamp("2021-12-31")
)

assert set(
    copula_assets
) == {
    "AGG",
    "GLD",
    "XLK"
}

assert len(
    copula_weights
) == 3

In [52]:
# Portfolio construction
assert np.isclose(
    copula_weights.sum(),
    1.0
)

# Copula simulations
assert (
    gaussian_uniform_sim > 0
).all()

assert (
    gaussian_uniform_sim < 1
).all()

assert (
    t_uniform_sim > 0
).all()

assert (
    t_uniform_sim < 1
).all()

# Tail-risk consistency
assert (
    copula_risk_results["ES"]
    >= copula_risk_results["VaR"]
).all()

# Student-t degrees of freedom
assert best_df > 2

# Three-model risk outputs
assert (
    three_model_comparison[
        [
            "VaR 95%",
            "VaR 99%",
            "ES 95%",
            "ES 99%"
        ]
    ]
    > 0
).all().all()

assert (
    three_model_comparison["ES 99%"]
    >= three_model_comparison["VaR 99%"]
).all()

# Model-fit preference
assert (
    copula_model_comparison.loc[
        copula_model_comparison["Model"]
        == "Student-t Copula",
        "Pseudo-AIC"
    ].iloc[0]
    <
    copula_model_comparison.loc[
        copula_model_comparison["Model"]
        == "Gaussian Copula",
        "Pseudo-AIC"
    ].iloc[0]
)
assert (
    np.linalg.eigvalsh(
        t_copula_corr.values
    )
    > 0
).all()

assert np.isclose(
    copula_weights.sum(),
    1.0,
    atol=1e-6
)

assert (
    np.isfinite(
        copula_weights
    )
    .all()
)
print(
    "All Notebook 09 final validation checks passed."
)

print(
    "\nPreferred dependence robustness model:"
    " Student-t Copula"
)

All Notebook 09 final validation checks passed.

Preferred dependence robustness model: Student-t Copula
